# Laboratorium 1: Dekoratory, Deskryptory i Generatory
### Skoroszyt główny

---

## Cele Laboratorium
Celem dzisiejszych zajęć jest opanowanie zaawansowanych konstrukcji języka Python, które są niezbędne do projektowania nowoczesnej architektury aplikacji.

### System Wspomagania AI (Tutor)
W trakcie rozwiązywania zadań możesz korzystać z pomocy dedykowanego tutora AI. System oferuje 6 poziomów wsparcia:
1. **Ogólna wskazówka**: Sugestia kierunku rozwiązania.
2. **Pseudokod**: Logiczny opis algorytmu.
3. **Mały fragment kodu**: Kluczowa linia lub konstrukcja.
4. **Częściowa implementacja**: Szkielet kodu do uzupełnienia.
5. **Szczegółowe wyjaśnienie**: Analiza mechanizmu działania.
6. **Pełne rozwiązanie**: Dostępne w sytuacjach ostatecznych.

---

## 1. Dekoratory

### DEMO: Dekorator @timer 
Stwórz dekorator @timer, który będzie mierzył i wyświetlał czas wykonania funkcji.

In [1]:
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"Czas wykonania {func.__name__}: {end_time - start_time:.4f} s")
        return result
    return wrapper

@timer
def example_task():
    time.sleep(0.5)
    print("Zadanie zakończone.")

example_task()

Zadanie zakończone.
Czas wykonania example_task: 0.5007 s


### Zadanie 1: Liczba elementów listy
Stwórz dekorator, który będzie odpowiedzialny za wyświetlanie liczby elementów listy, jeśli jakakolwiek lista pojawi się w parametrach funkcji dekorowanej. 

**Protip:** użyj isinstance do sprawdzenia czy parametr jest listą. Pamiętaj o zachowaniu metadanych funkcji.

In [2]:
import functools

# TODO: Implementacja dekoratora
def show_list_length(func):
    @functools.wraps(func)
    def wrapper(*args):
        if isinstance(args[0], list):
            return len(args[0])
    return wrapper
        
        
# Test:
@show_list_length
def process_data(data_list, name):
    print(f"Przetwarzanie {name}")

process_data([1,5,6,7,8,9,9], "Test")

7

### Zadanie 2: Logowanie do pliku
Stwórz dekorator, który będzie zapisywał w pliku *.log nazwę funkcji dekorowanej, datę oraz długość wykonania. Nazwa pliku będzie podana jako argument dekoratora.

**Protip:** użyj biblioteki datetime. Pamiętaj o tym, żeby dekoratory przyjęły metadanych funkcji dekorującej.

In [ ]:
import functools
from datetime import datetime
import time

# TODO: Implementacja dekoratora z argumentem
def logger(func, filename = "data.log"):
    functools.wraps(func)
    date=datetime.today().strftime("%Y-%m-%d")
    start_time=time.time()
    func()
    exec_time=time.time()-start_time
    def wrapper(*args):
        log_file = open(filename, "w")
        log_file.write(func.__name__ + "\n")
        log_file.write(date + "\n")
        log_file.write(str(exec_time))
        log_file.close()
    return wrapper
    
@logger
def test_function():
    print("Test")
    
test_function()    

Test


--- 
## 2. Deskryptory

### DEMO: Walidator e-mail klasy Student
Stwórz deskryptor, który będzie działał jako walidator email klasy Student. Klasa Student zawiera pola imie, nazwisko i email. Deskryptor ten powinien sprawdzać poprawność danych wprowadzanych podczas tworzenia lub modyfikowania instancji Student.

In [25]:
class EmailValidator:
    def __set_name__(self, owner, name):
        self.name = name

    def __set__(self, instance, value):
        if "@" not in value:
            raise ValueError(f"Błędny format adresu email: {value}")
        instance.__dict__[self.name] = value

class Student:
    email = EmailValidator()
    
    def __init__(self, imie, nazwisko, email):
        self.imie = imie
        self.nazwisko = nazwisko
        self.email = email

try:
    s = Student("Jan", "Kowalski", "jan.kowalskiwsei.edu.pl")
    print(f"Utworzono studenta: {s.email}")
    # s.email = "invalid_at_email" # Powinno rzucić błąd
except ValueError as e:
    print(e)

Błędny format adresu email: jan.kowalskiwsei.edu.pl


### Zadanie 3: Rejestrowanie dostępu
Stwórz klasę Uzytkownik. Klasa powinna zawierać atrybuty imie i wiek. Opracuj deskryptor, który będzie rejestrował dostęp do tych atrybutów za pomocą logowania. Deskryptor powinien logować informacje o odczycie (__get__) oraz zapisie (__set__) wartości atrybutu.

In [56]:
# TODO: Implementacja deskryptora logującego dostęp
class AccessLogger:
    
    def __init__(self, name):
        self.name = name
    
    def __set__(self, instance, value):
        instance.__dict__[self.name] = value
        log = open("log.txt", "a")
        log.write(f"Uzytkownik ustawil atrybut {self.name} na {value} \n")
        log.close()
    
    def __get__(self, instance, owner):
        log = open("log.txt", "a")
        log.write(f"Uzytkownik {instance.__dict__["imie"]} odczytal atrybut {self.name} \n")
        log.close()
        return instance.__dict__[self.name]
        

class Uzytkownik:
    imie = AccessLogger("imie")
    wiek = AccessLogger("wiek")
    
    def __init__(self, imie, wiek):
        self.imie = imie
        self.wiek = wiek
        
        self.get_imie()
        self.get_wiek()
    
    def get_imie(self):
        return self.imie
    
    def get_wiek(self):
        return self.wiek
    
        
try:
    u = Uzytkownik("Adam", 16)
    
except ValueError as e:
    print(f"Error {e}")

--- 
## 3. Generatory i Iteratory

### DEMO: Generator Fibonacciego
Napisz klasę, która będzie implementowała generator ciągu Fibonacciego za pomocą metod magicznych __iter__() i __next__().

In [57]:
class FibonacciGenerator:
    def __init__(self, limit):
        self.limit = limit
        self.a, self.b = 0, 1
        self.count = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.count >= self.limit:
            raise StopIteration
        
        result = self.a
        self.a, self.b = self.b, self.a + self.b
        self.count += 1
        return result

fib = FibonacciGenerator(10)
print(list(fib))

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


### Zadanie 4: Generator ciągu Collatza
Opracuj generator ciągu Collatza. Dla liczby naturalnej n, jeśli n jest parzyste, dziel przez 2; jeśli n jest nieparzyste, pomnóż przez 3 i dodaj 1, zaczynając od określonej liczby początkowej, aż do osiągnięcia wartości 1.

In [ ]:
# TODO: Implementacja generatora ciągu Collatza
collatz_sequence = []
def collatz_generator(n):
    while n > 1:
        if n%2==0:
            collatz_sequence.append(int(n/2))
            print(n/2)
            return collatz_generator(n/2)
        else:
            collatz_sequence.append(int((n*3) + 1))
            print(n*3 + 1)
            return collatz_generator((n*3) + 1)
    pass

# Test:
for status in collatz_generator(10):
    print(status)
#sequence = collatz_generator(10)
#print(sequence)

5.0
16.0
8.0
4.0
2.0
1.0


TypeError: 'NoneType' object is not iterable

---

## Zadania do zrobienia w domu

Poniższe zadania stanowią rozszerzenie materiału i są przeznaczone dla osób chcących zgłębić temat zaawansowanych konstrukcji języka Python.

### Zadanie dodatkowe 1: Dekorator z autoryzacją
Stwórz dekorator `@require_role(role)`, który przyjmuje nazwę wymaganej roli jako argument. Dekorator powinien sprawdzać, czy w globalnym słowniku `current_user` klucz `role` jest zgodny z wymaganym. Jeśli nie, rzuć `PermissionError`.

In [84]:
current_user = {"username": "admin", "role": "superuser"}

# TODO: Implementacja dekoratora @require_role
def require_role(role):
    functools.wraps(role)
    def wrapper(*args):
        if args[1] != current_user["role"]:
            raise PermissionError
    return wrapper

@require_role
def check_permission(username, role):
    user = {"username": username, "role": role}
    return user
        
check_permission("test", "superuser")

### Zadanie dodatkowe 2: Deskryptor z walidacją typu
Stwórz deskryptor `Typed`, który przyjmuje typ danych (np. `int`, `str`) w konstruktorze. Deskryptor powinien upewnić się, że zapisywana wartość jest tego typu. Jeśli nie, rzuć `TypeError`.

In [105]:
# TODO: Implementacja deskryptora Typed
class Typed:
    
    def __init__(self, val):
        self.val=val
    
    def __set__(self, instance, value):
        if self.val=="description" and isinstance(value, str):
            instance.__dict__[self.val] = value
        elif isinstance(value, int):
            instance.__dict__[self.val] = value
        else:
            raise TypeError
        
class Grade:
    grade = Typed("grade")
    description = Typed("description")
    def __init__(self, grade, description):
        self.grade=grade
        self.description=description
   
try:    
    grade = Grade(4, "Final lab grade")
except ValueError as e:
    print(f"Error {e}")
    

### Zadanie dodatkowe 3: Nieskończony generator liczb pierwszych
Opracuj generator `prime_generator`, który zwraca kolejne liczby pierwsze. Następnie użyj wyrażenia generatorowego, aby stworzyć iterator zwracający tylko te liczby pierwsze, które kończą się cyfrą 7.

In [147]:
# TODO: Implementacja generatora liczb pierwszych
number = 1
def prime_generator():
    global number
    while(number < 1000):
        for i in range(1, number+1):
            prime = True
            if number == 1:
                prime = False
            if i!=number and i!=1:
                if number % i == 0:
                    prime = False
                    break
        if prime:
            yield number
        number+=1
                
for prime in prime_generator():
    string_number=str(prime)
    if string_number[-1]=="7":
        print(prime)

7
17
37
47
67
97
107
127
137
157
167
197
227
257
277
307
317
337
347
367
397
457
467
487
547
557
577
587
607
617
647
677
727
757
787
797
827
857
877
887
907
937
947
967
977
997
